# DWDM Mini Project: Customer Shopping Behaviour Analysis using Market Basket Analysis
**Tools:** Python + WEKA  |  **Team:** Individual  |  **Algorithm:** Apriori (with hash tree counting)

> The dataset is **synthetic** (generated for educational use). Mention this in your report.

Run the cells from top to bottom. Keep `customer_shopping_transactions.csv` in the same folder as this notebook.

In [2]:
pip install mlxtend

Note: you may need to restart the kernel to use updated packages.


## Step 1: Setup

In [3]:
# Run once if a library is missing:
# !pip install pandas numpy mlxtend
import pandas as pd
import numpy as np
from itertools import combinations
from collections import Counter
print("pandas", pd.__version__)

pandas 2.3.3


## Step 2: Load and explore the raw dataset

In [4]:
df = pd.read_csv("customer_shopping_transactions.csv")
print("Shape before cleaning:", df.shape)
print(df.head())
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("Rows with quantity <= 0:", (df["quantity"] <= 0).sum())
print("Rows with unit_price <= 0:", (df["unit_price"] <= 0).sum())

Shape before cleaning: (2404, 12)
  transaction_id customer_id transaction_date age_group gender      city  \
0          T0001        C007       2025-11-01       50+   Male  Hubballi   
1          T0001        C007       2025-11-01       50+   Male  Hubballi   
2          T0001        C007       2025-11-01       50+   Male  Hubballi   
3          T0001        C007       2025-11-01       50+   Male  Hubballi   
4          T0002        C075       2025-05-16     18-25   Male  Hubballi   

  product    category  quantity  unit_price payment_method discount_applied  
0   Onion  Vegetables         2          35           Cash              Yes  
1    Eggs       Dairy         1          70           Cash              Yes  
2   Bread      Bakery         3          40           Cash              Yes  
3  Potato  Vegetables         1          28           Cash              Yes  
4   Sugar     Grocery         3          45            UPI              Yes  

Missing values per column:
transaction_i

## Step 3: Preprocessing

In [5]:
before = len(df)
df = df.drop_duplicates()                                   # 1. remove duplicates
df = df.dropna(subset=["transaction_id", "product"])        # 2. no transaction id / product
df["city"] = df["city"].fillna("Unknown")                   # 3. fill missing city
df = df[(df["quantity"] > 0) & (df["unit_price"] > 0)]      # 4. remove invalid values

# 5. derived columns
df["transaction_date"] = pd.to_datetime(df["transaction_date"])
df["total_amount"] = df["quantity"] * df["unit_price"]
df["month"] = df["transaction_date"].dt.month_name()
df["quarter"] = "Q" + df["transaction_date"].dt.quarter.astype(str)
df["year"] = df["transaction_date"].dt.year

# keep months in calendar order (not alphabetical)
months = ["January","February","March","April","May","June","July",
          "August","September","October","November","December"]
df["month"] = pd.Categorical(df["month"], categories=months, ordered=True)

print(f"Rows before: {before}, after: {len(df)}, removed: {before - len(df)}")
print("Transactions:", df["transaction_id"].nunique(), "| Products:", df["product"].nunique())
df.to_csv("cleaned_transactions.csv", index=False)

Rows before: 2404, after: 2398, removed: 6
Transactions: 601 | Products: 20


## Step 4: Star schema (multidimensional model)
One fact table (`Fact_Sales`) surrounded by four dimension tables.

In [6]:
dim_customer = df[["customer_id", "age_group", "gender", "city"]].drop_duplicates().reset_index(drop=True)
dim_customer["customer_key"] = dim_customer.index + 1

dim_product = df[["product", "category"]].drop_duplicates().reset_index(drop=True)
dim_product["product_key"] = dim_product.index + 1

dim_date = df[["transaction_date", "month", "quarter", "year"]].drop_duplicates().reset_index(drop=True)
dim_date["date_key"] = dim_date.index + 1

dim_payment = df[["payment_method", "discount_applied"]].drop_duplicates().reset_index(drop=True)
dim_payment["payment_key"] = dim_payment.index + 1

fact_sales = (df.merge(dim_customer)
                .merge(dim_product)
                .merge(dim_date)
                .merge(dim_payment)
                [["customer_key", "product_key", "date_key", "payment_key",
                  "transaction_id", "quantity", "unit_price", "total_amount"]])
fact_sales.insert(0, "sales_key", range(1, len(fact_sales) + 1))

print("Fact_Sales:", fact_sales.shape)
print(fact_sales.head())
print("Dim_Customer:", dim_customer.shape, "| Dim_Product:", dim_product.shape,
      "| Dim_Date:", dim_date.shape, "| Dim_Payment:", dim_payment.shape)

Fact_Sales: (2398, 9)
   sales_key  customer_key  product_key  date_key  payment_key transaction_id  \
0          1             1            1         1            1          T0001   
1          2             1            2         1            1          T0001   
2          3             1            3         1            1          T0001   
3          4             1            4         1            1          T0001   
4          5             2            5         2            2          T0002   

   quantity  unit_price  total_amount  
0         2          35            70  
1         1          70            70  
2         3          40           120  
3         1          28            28  
4         3          45           135  
Dim_Customer: (82, 5) | Dim_Product: (20, 3) | Dim_Date: (297, 5) | Dim_Payment: (6, 3)


       ┌────────────────────────┐             ┌────────────────────────┐
       │      DIM_CUSTOMER      │             │      DIM_PRODUCT       │
       ├────────────────────────┤             ├────────────────────────┤
       │ PK │ customer_key      │             │ PK │ product_key       │
       │    │ customer_id       │             │    │ product           │
       │    │ age_group         │             │    │ category          │
       │    │ gender            │             └───────────┬────────────┘
       │    │ city              │                         │
       └───────────┬────────────┘                         │
                   │                                      │
                   │            ┌────────────┐            │
                   └───────────►│ FACT_SALES │◄───────────┘
                                ├────────────┤
                                │ sales_key  │ ◄── Primary Key
                                │            │
                                │customer_key│ ◄── Foreign Key (Customer)
                                │product_key │ ◄── Foreign Key (Product)
                                │date_key    │ ◄── Foreign Key (Date)
                                │payment_key │ ◄── Foreign Key (Payment)
                                │            │
                   ┌───────────►│transaction_│ ◄── Degenerate Dimension
                   │            │id          │
                   │            │            │
                   │            │quantity    │ ◄── Measure
                   │            │unit_price  │ ◄── Measure
                   │            │total_amount│ ◄── Measure
                   │            └────────────┘            │
       ┌───────────┴────────────┐             ┌───────────┴────────────┐
       │        DIM_DATE        │             │      DIM_PAYMENT       │
       ├────────────────────────┤             ├────────────────────────┤
       │ PK │ date_key          │             │ PK │ payment_key       │
       │    │ transaction_date  │             │    │ payment_method    │
       │    │ month             │             │    │ discount_applied  │
       │    │ quarter           │             └────────────────────────┘
       │    │ year              │
       └────────────────────────┘

## Step 5: Data cube and OLAP operations
Cube = **city x category x quarter**, measure = total sales amount.

In [7]:
# The data cube
cube = df.pivot_table(index=["city", "category"], columns="quarter",
                      values="total_amount", aggfunc="sum", fill_value=0, observed=True)
print(cube)

quarter                     Q1    Q2    Q3    Q4
city       category                             
Ballari    Bakery         1000   600  1080   720
           Beverages      4990  2505  4215  3720
           Dairy          2800  1725  2430  2070
           Grocery        4190  3180  3815  7620
           Household       660   110   550   770
           Personal Care  2490  3385  2575  1780
           Snacks          730   530   870   595
           Vegetables     1809   853  2389  1302
Bengaluru  Bakery          800  1280   960   960
           Beverages      5205  3735  3925  1710
           Dairy          2020  2780  1830  1375
           Grocery        6080  3790  3105  3385
           Household         0   330     0   660
           Personal Care  3080  1850  2160   990
           Snacks          695   335   515   215
           Vegetables     1065   542   668   934
Hubballi   Bakery          520  1680   920   960
           Beverages      4275  8420  5500  4810
           Dairy    

In [8]:
# ROLL-UP: month -> quarter -> year (less detail, more summary)
print("Level 1 (month):")
print(df.groupby(["year", "quarter", "month"], observed=True)["total_amount"].sum().head(12))
print("\nLevel 2 (quarter):")
print(df.groupby(["year", "quarter"], observed=True)["total_amount"].sum())
print("\nLevel 3 (year):")
print(df.groupby("year", observed=True)["total_amount"].sum())

Level 1 (month):
year  quarter  month    
2025  Q1       January      21964
               February     28245
               March        26709
      Q2       April        20173
               May          27999
               June         34348
      Q3       July         27070
               August       27839
               September    21726
      Q4       October      21903
               November     28306
               December     18205
Name: total_amount, dtype: int64

Level 2 (quarter):
year  quarter
2025  Q1         76918
      Q2         82520
      Q3         76635
      Q4         68414
Name: total_amount, dtype: int64

Level 3 (year):
year
2025    304487
Name: total_amount, dtype: int64


In [9]:
# DRILL-DOWN: the reverse of roll-up. Start at year, go down to quarter, then month.
year_total = df.groupby("year")["total_amount"].sum()
q_total = df.groupby(["year", "quarter"])["total_amount"].sum()
print("Year ->", year_total.to_dict())
print("Drill down to quarters ->")
print(q_total)
print("Drill down into Q1 months ->")
print(df[df["quarter"] == "Q1"].groupby("month", observed=True)["total_amount"].sum())

Year -> {2025: 304487}
Drill down to quarters ->
year  quarter
2025  Q1         76918
      Q2         82520
      Q3         76635
      Q4         68414
Name: total_amount, dtype: int64
Drill down into Q1 months ->
month
January     21964
February    28245
March       26709
Name: total_amount, dtype: int64


In [10]:
# SLICE: fix ONE dimension (city = Ballari)
print(df[df["city"] == "Ballari"].groupby("category")["total_amount"].sum())

category
Bakery            3400
Beverages        15430
Dairy             9025
Grocery          18805
Household         2090
Personal Care    10230
Snacks            2725
Vegetables        6353
Name: total_amount, dtype: int64


In [11]:
# DICE: select values on TWO or more dimensions
dice = df[df["city"].isin(["Ballari", "Raichur"]) & df["category"].isin(["Dairy", "Grocery"])]
print(dice.pivot_table(index="city", columns="category",
                       values="total_amount", aggfunc="sum"))

category  Dairy  Grocery
city                    
Ballari    9025    18805
Raichur    7025    14815


## Step 6: CSV to ARFF (basket form for WEKA)
Each row = one transaction, each column = one product.
Value `t` = bought, `?` = not bought. Using `{t}` only stops WEKA's Apriori from treating "not bought" as an item.

In [12]:
items = sorted(df["product"].unique())
baskets = df.groupby("transaction_id")["product"].apply(set)

with open("shopping_baskets.arff", "w") as f:
    f.write("@RELATION shopping_baskets\n\n")
    for it in items:
        f.write(f"@ATTRIBUTE '{it}' {{t}}\n")
    f.write("\n@DATA\n")
    for b in baskets:
        f.write(",".join("t" if it in b else "?" for it in items) + "\n")

print("ARFF written:", len(baskets), "transactions,", len(items), "product attributes")

ARFF written: 601 transactions, 20 product attributes


## Step 7: Apriori in Python (mlxtend)

In [13]:
from mlxtend.frequent_patterns import apriori, association_rules

basket = (df.assign(present=1)
            .pivot_table(index="transaction_id", columns="product",
                         values="present", aggfunc="max", fill_value=0, observed=True)
            .astype(bool))

MIN_SUPPORT = 0.05
MIN_CONF = 0.30

frequent_itemsets = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True)
print("Frequent itemsets:", len(frequent_itemsets))

try:
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=MIN_CONF)
except TypeError:          # newer mlxtend versions need num_itemsets
    rules = association_rules(frequent_itemsets, num_itemsets=len(basket),
                              metric="confidence", min_threshold=MIN_CONF)

rules = rules.sort_values("lift", ascending=False)
print("Rules:", len(rules))
print(rules[["antecedents", "consequents", "support", "confidence", "lift"]].head(10))

rules.to_csv("association_rules_output.csv", index=False)
frequent_itemsets.to_csv("frequent_itemsets_output.csv", index=False)

Frequent itemsets: 50
Rules: 76
          antecedents        consequents   support  confidence      lift
69           (Potato)    (Tomato, Onion)  0.104825    0.552632  4.813501
64    (Tomato, Onion)           (Potato)  0.104825    0.913043  4.813501
58        (Dal, Rice)      (Cooking Oil)  0.093178    0.811594  4.690078
63      (Cooking Oil)        (Dal, Rice)  0.093178    0.538462  4.690078
44              (Tea)  (Sugar, Biscuits)  0.111481    0.670000  4.682209
41  (Sugar, Biscuits)              (Tea)  0.111481    0.779070  4.682209
67           (Tomato)    (Potato, Onion)  0.104825    0.547826  4.637232
66    (Potato, Onion)           (Tomato)  0.104825    0.887324  4.637232
45         (Biscuits)       (Sugar, Tea)  0.111481    0.563025  4.635317
40       (Sugar, Tea)         (Biscuits)  0.111481    0.917808  4.635317


In [14]:
# Experiment: how many rules for different thresholds? (use this table in your report)
results = []
for s in [0.03, 0.05, 0.08, 0.10]:
    for c in [0.3, 0.5, 0.7]:
        fi = apriori(basket, min_support=s, use_colnames=True)
        try:
            r = association_rules(fi, metric="confidence", min_threshold=c)
        except TypeError:
            r = association_rules(fi, num_itemsets=len(basket), metric="confidence", min_threshold=c)
        results.append({"min_support": s, "min_confidence": c,
                        "frequent_itemsets": len(fi), "rules": len(r)})
print(pd.DataFrame(results))

    min_support  min_confidence  frequent_itemsets  rules
0          0.03             0.3                108     82
1          0.03             0.5                108     65
2          0.03             0.7                108     27
3          0.05             0.3                 50     76
4          0.05             0.5                 50     59
5          0.05             0.7                 50     24
6          0.08             0.3                 45     76
7          0.08             0.5                 45     59
8          0.08             0.7                 45     24
9          0.10             0.3                 43     64
10         0.10             0.5                 43     49
11         0.10             0.7                 43     18


## Step 8: Apriori in WEKA (do this in the GUI)
1. Open **WEKA GUI Chooser -> Explorer**.
2. **Preprocess** tab -> Open file -> `shopping_baskets.arff`.
3. **Associate** tab -> Choose -> `weka.associations.Apriori`.
4. Click the Apriori name and set: `lowerBoundMinSupport = 0.05`, `minMetric = 0.3`, `metricType = Confidence`, `numRules = 500` (large, so Weka does not stop early and can be compared with Python).
5. Click **Start** and take a screenshot of the output.

## Step 9: Hash tree for candidate counting
Apriori must count how many transactions contain each candidate itemset. A **hash tree** stores candidates in leaves; each transaction is compared only with the leaves it can reach instead of every candidate.

In [15]:
class Node:
    def __init__(self):
        self.children = {}
        self.items = []        # entries: [itemset_tuple, count]
        self.is_leaf = True

class HashTree:
    def __init__(self, k, buckets=3, max_leaf=3, hash_fn=None):
        self.k, self.buckets, self.max_leaf = k, buckets, max_leaf
        self.hash_fn = hash_fn or (lambda item: sum(map(ord, item)))
        self.root = Node()

    def _hash(self, item):
        return self.hash_fn(item) % self.buckets

    def insert(self, itemset):
        self._insert(self.root, [tuple(itemset), 0], 0)

    def _insert(self, node, entry, depth):
        if not node.is_leaf:
            h = self._hash(entry[0][depth])
            self._insert(node.children.setdefault(h, Node()), entry, depth + 1)
            return
        node.items.append(entry)
        if len(node.items) > self.max_leaf and depth < self.k:
            node.is_leaf = False
            old, node.items = node.items, []
            for e in old:
                h = self._hash(e[0][depth])
                self._insert(node.children.setdefault(h, Node()), e, depth + 1)

    def count(self, transaction):
        self._visit(self.root, sorted(transaction), 0, set())

    def _visit(self, node, t, start, seen):
        if node.is_leaf:
            if id(node) in seen:
                return
            seen.add(id(node))
            ts = set(t)
            for e in node.items:
                if set(e[0]) <= ts:
                    e[1] += 1
            return
        for i in range(start, len(t)):
            h = self._hash(t[i])
            if h in node.children:
                self._visit(node.children[h], t, i + 1, seen)

    def results(self, node=None):
        node = node or self.root
        if node.is_leaf:
            return [(e[0], e[1]) for e in node.items]
        return [r for c in node.children.values() for r in self.results(c)]

    def show(self, node=None, indent=0, label="root"):
        node = node or self.root
        pad = "    " * indent
        if node.is_leaf:
            print(f"{pad}[{label}] leaf: {[list(e[0]) for e in node.items]}")
        else:
            print(f"{pad}[{label}] internal node")
            for h in sorted(node.children):
                self.show(node.children[h], indent + 1, f"hash {h}")

In [16]:
# Worked example (textbook style): 15 candidate 3-itemsets, hash function = item mod 3
cands = ["145","124","457","125","458","159","136","234","567","345","356","357","689","367","368"]
demo = HashTree(k=3, buckets=3, max_leaf=3, hash_fn=lambda x: int(x))
for c in cands:
    demo.insert(tuple(c))          # e.g. "145" -> ('1','4','5')
demo.show()

demo.count(["1","2","3","5","6"])  # one transaction {1,2,3,5,6}
print("\nCandidates contained in transaction {1,2,3,5,6}:")
print([(''.join(s), c) for s, c in demo.results() if c > 0])

[root] internal node
    [hash 0] internal node
        [hash 0] leaf: [['3', '6', '7'], ['3', '6', '8']]
        [hash 1] leaf: [['3', '4', '5']]
        [hash 2] leaf: [['3', '5', '6'], ['3', '5', '7'], ['6', '8', '9']]
    [hash 1] internal node
        [hash 0] leaf: [['1', '3', '6']]
        [hash 1] leaf: [['1', '4', '5']]
        [hash 2] internal node
            [hash 0] leaf: [['1', '5', '9']]
            [hash 1] leaf: [['1', '2', '4'], ['4', '5', '7']]
            [hash 2] leaf: [['1', '2', '5'], ['4', '5', '8']]
    [hash 2] leaf: [['2', '3', '4'], ['5', '6', '7']]

Candidates contained in transaction {1,2,3,5,6}:
[('125', 1), ('136', 1), ('356', 1)]


In [17]:
# Use the hash tree for Apriori pass 2 (candidate 2-itemsets) on the real data
transactions = [sorted(b) for b in baskets]
n = len(transactions)

c1 = Counter(i for t in transactions for i in set(t))
L1 = sorted(i for i, c in c1.items() if c / n >= MIN_SUPPORT)

tree = HashTree(k=2)
for cand in combinations(L1, 2):
    tree.insert(cand)
for t in transactions:
    tree.count(t)

L2 = [(s, c, c / n) for s, c in tree.results() if c / n >= MIN_SUPPORT]
print("Candidate 2-itemsets:", len(list(combinations(L1, 2))), "| Frequent 2-itemsets:", len(L2))
for s, c, sup in sorted(L2, key=lambda x: -x[1])[:10]:
    print(s, c, round(sup, 3))

Candidate 2-itemsets: 190 | Frequent 2-itemsets: 24
('Bread', 'Milk') 97 0.161
('Milk', 'Sugar') 92 0.153
('Bread', 'Eggs') 90 0.15
('Butter', 'Milk') 87 0.145
('Biscuits', 'Sugar') 86 0.143
('Coffee', 'Sugar') 83 0.138
('Coffee', 'Milk') 81 0.135
('Potato', 'Tomato') 80 0.133
('Bread', 'Butter') 75 0.125
('Biscuits', 'Tea') 74 0.123


In [18]:
# Verification: hash tree counts must match mlxtend's frequent 2-itemsets
ml2 = {frozenset(r["itemsets"]): r["support"]
       for _, r in frequent_itemsets.iterrows() if len(r["itemsets"]) == 2}
ht2 = {frozenset(s): sup for s, c, sup in L2}
print("Same set of frequent 2-itemsets:", set(ml2) == set(ht2))
print("Same support values:", all(abs(ml2[k] - ht2[k]) < 1e-9 for k in ml2 if k in ht2))

Same set of frequent 2-itemsets: True
Same support values: True


## Step 10: Summary for your report
Fill your Results chapter with your own output: top 5 rules (support, confidence, lift) with a one-line business meaning each, a Weka vs Python comparison, and the threshold table from Step 7.